# 16 Market-Calibrated Volatility Surface

Use real CSV data if available, otherwise synthetic Indian IV data, to interpolate smiles, term structure, and a volatility surface.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
processed = PROCESSED_DATA_DIR / "normalized_option_chain.csv"
if processed.exists():
    chain = pd.read_csv(processed)
else:
    chain = generate_synthetic_indian_market(config)
chain = normalize_option_chain_columns(chain)
iv_data = chain.dropna(subset=["strike", "maturity", "implied_volatility", "underlying_price"]).copy()
iv_data = iv_data[(iv_data["symbol"] == "NIFTY") & (iv_data["option_type"] == "call")]
iv_data["moneyness"] = iv_data["strike"] / iv_data["underlying_price"]
save_table(iv_data[["symbol", "strike", "maturity", "moneyness", "implied_volatility"]], "16_iv_surface_input.csv")
plt.figure()
for maturity, group in iv_data.groupby("maturity"):
    plt.plot(group.sort_values("moneyness")["moneyness"], group.sort_values("moneyness")["implied_volatility"], marker="o", label=f"{maturity*365:.0f}d")
plt.title("Volatility smile by maturity")
plt.xlabel("Strike / spot")
plt.ylabel("Implied volatility")
plt.legend()
save_current_figure("16_volatility_smile.png")
term = iv_data.groupby("maturity")["implied_volatility"].mean().reset_index()
plt.figure()
plt.plot(term["maturity"] * 365, term["implied_volatility"], marker="o")
plt.title("Average volatility term structure")
plt.xlabel("Days to expiry")
plt.ylabel("Average implied volatility")
save_current_figure("16_volatility_term_structure.png")
mx = np.linspace(iv_data["moneyness"].min(), iv_data["moneyness"].max(), 40)
mt = np.linspace(iv_data["maturity"].min(), iv_data["maturity"].max(), 30)
MX, MT = np.meshgrid(mx, mt)
points = iv_data[["moneyness", "maturity"]].values
values = iv_data["implied_volatility"].values
IV = griddata(points, values, (MX, MT), method="linear")
IV_nearest = griddata(points, values, (MX, MT), method="nearest")
IV = np.where(np.isnan(IV), IV_nearest, IV)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(MX, MT * 365, IV, cmap="viridis", alpha=0.9)
ax.set_title("Interpolated implied volatility surface")
ax.set_xlabel("Strike / spot")
ax.set_ylabel("Days")
ax.set_zlabel("IV")
save_current_figure("16_volatility_surface.png")
# Use interpolated IV for a sample pricing table.
sample = iv_data.head(12).copy()
sample["bs_price_with_market_iv"] = [black_scholes_price(row.underlying_price, row.strike, row.maturity, 0.065, row.implied_volatility, "call") for row in sample.itertuples()]
save_table(sample, "16_market_iv_pricing_sample.csv")
sample.head()
